# Notebook C — Multiple Kernel Learning : théorie et méthodes

Comment combiner plusieurs kernels ? Pourquoi ça aide ? Quand est-ce que ça aide ?

**Plan :**
1. Le problème du choix de kernel — no-free-lunch
2. La formulation MKL — preuve que c'est un kernel valide
3. Average — la baseline naïve et ses limites
4. Centered Alignment — dérivation complète pas à pas
5. SDP — formulation exacte
6. Quand MKL bat-il un kernel unique ? Condition de complémentarité


## Section 1 — Le problème du choix de kernel

### Pourquoi choisir un kernel est difficile

Un kernel $K$ est un **a priori sur la structure des données** :
- RBF suppose que des points proches (au sens euclidien) sont similaires
- Kernel cosinus (ZZ) suppose que des patterns périodiques sont importants
- Polynomial suppose des interactions polynomiales entre features

Si cet a priori est faux, les performances chutent. Or, on ne sait généralement pas à l'avance quelle structure ont les données financières.

**Théorème No-Free-Lunch (Wolpert, 1996) :** Aucun algorithme d'apprentissage n'est universellement meilleur. Pour tout algorithme $A$ qui performe bien sur une distribution, il existe une distribution sur laquelle $A$ performe mal.

**Conséquence pour les kernels :** Aucun kernel ne domine tous les autres sur tous les datasets. Empiriquement dans notre projet :
- Sur German Credit : RBF-SVM $= 0.800$, kernel Z $= 0.712$, kernel ZZ $= 0.756$
- Sur Bank Marketing : Les différences entre kernels sont plus faibles
- L'ordre des kernels **change selon le dataset**

### La solution MKL

Au lieu de choisir un kernel, on **combine** plusieurs kernels :
$$K_{\text{mkl}}(x, x') = \sum_{m=1}^M w_m K_m(x, x')$$

avec $w_m \geq 0$ et $\sum_m w_m = 1$.

Maintenant la question est : **comment trouver les bons poids $w_m$ ?**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.datasets import make_moons, make_circles, make_classification
from sklearn.preprocessing import StandardScaler, MinMaxScaler

np.random.seed(42)

# ── Montrer que l'ordre des kernels dépend du dataset ────────────────────
def K_rbf(X, gamma=1.0):
    sq = np.sum(X**2, axis=1, keepdims=True)
    return np.exp(-gamma * (sq + sq.T - 2*(X@X.T)))

def K_poly(X, p=3, c=1.0):
    return (X@X.T/X.shape[1] + c)**p

def K_Z(X, alpha=1.0):
    n, d = X.shape
    K = np.ones((n,n))
    for k in range(d):
        diff = X[:,k:k+1] - X[:,k].reshape(1,-1)
        K *= np.cos(alpha*diff)**2
    return K

def K_ZZ(X, alpha=1.0):
    K = K_Z(X, alpha)
    n, d = X.shape
    for k in range(d):
        for l in range(k+1, d):
            c = X[:,k]*X[:,l]
            diff = c.reshape(-1,1) - c.reshape(1,-1)
            K *= np.cos(alpha*diff)**2
    return K

# 4 datasets avec structures différentes
datasets = {
    'Gaussien (linéaire)': make_classification(n_samples=150, n_features=4,
                           n_informative=2, n_redundant=0, random_state=42),
    'Cercles (circulaire)': make_circles(n_samples=150, noise=0.1, factor=0.4, random_state=42),
    'Lunes (non-linéaire)': make_moons(n_samples=150, noise=0.15, random_state=42),
    'Damier (haute fréq.)': None,  # généré ci-dessous
}
# Damier = XOR multi-fréquence
X_d = np.random.uniform(0, 2, (150, 2))
y_d = ((np.floor(X_d[:,0]*3).astype(int) + np.floor(X_d[:,1]*3).astype(int)) % 2).astype(int)
datasets['Damier (haute fréq.)'] = (X_d, y_d)

kernels = [
    ('RBF γ=1',  lambda X: K_rbf(X, 1.0)),
    ('Poly p=3',  lambda X: K_poly(X, 3)),
    ('Z α=1.0',  lambda X: K_Z(X, 1.0)),
    ('ZZ α=1.0', lambda X: K_ZZ(X, 1.0)),
    ('ZZ α=2.5', lambda X: K_ZZ(X, 2.5)),
]

fig, ax = plt.subplots(figsize=(11, 5))
x_pos = np.arange(len(datasets))
width = 0.15
colors = ['#3498db','#e74c3c','#2ecc71','#f39c12','#9b59b6']

for k_idx, (k_name, k_fn) in enumerate(kernels):
    aucs = []
    for ds_name, (X, y) in datasets.items():
        scaler = MinMaxScaler(feature_range=(0,2))
        X_s = scaler.fit_transform(StandardScaler().fit_transform(X))
        if X_s.shape[1] < 4:
            X_s = np.hstack([X_s, np.zeros((len(X_s), 4-X_s.shape[1]))])
        X_s = X_s[:, :4]
        K = k_fn(X_s)
        # Fixer PSD
        eigs = np.linalg.eigvalsh(K)
        if eigs.min() < 0:
            K += (-eigs.min() + 1e-8)*np.eye(len(K))
        y_pm = 2*y - 1
        try:
            auc = cross_val_score(SVC(kernel='precomputed', C=1.0), K, y_pm,
                                  cv=5, scoring='roc_auc').mean()
        except:
            auc = 0.5
        aucs.append(auc)
    ax.bar(x_pos + k_idx*width, aucs, width, label=k_name, color=colors[k_idx], alpha=0.85)

ax.set_xticks(x_pos + width*2)
ax.set_xticklabels(list(datasets.keys()), fontsize=9)
ax.set_ylabel('AUC (cross-val 5-fold)')
ax.set_title("L'ordre des kernels dépend du dataset\n"
             "→ No-Free-Lunch : aucun kernel ne domine toujours", fontweight='bold')
ax.legend(fontsize=8, loc='lower right')
ax.set_ylim(0.4, 1.05)
ax.axhline(0.5, color='grey', ls=':', lw=1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("Observation: aucun kernel n'est meilleur sur tous les datasets.")
print("→ Motivation pour MKL : combiner les kernels pour être robuste.")


## Section 2 — Multiple Kernel Learning : formulation et propriétés

### La combinaison linéaire de kernels

**Théorème :** Si $K_1, ..., K_M$ sont des kernels valides (PSD) et $w_1, ..., w_M \geq 0$, alors :
$$K_{\text{mkl}} = \sum_{m=1}^M w_m K_m$$
est aussi un kernel valide.

**Preuve :** Pour tout vecteur $\mathbf{v}$ :
$$\mathbf{v}^\top \mathbf{K}_{\text{mkl}} \mathbf{v} = \sum_m w_m \underbrace{\mathbf{v}^\top \mathbf{K}_m \mathbf{v}}_{\geq 0 \text{ (PSD)}} \geq 0 \quad \text{si } w_m \geq 0$$

**RKHS associé :** Le RKHS de $K_{\text{mkl}}$ est l'espace produit pondéré :
$$\mathcal{H}_{\text{mkl}} = \left\{f = \sum_m f_m \;\middle|\; f_m \in \mathcal{H}_m \right\}, \quad \|f\|^2_{\mathcal{H}_{\text{mkl}}} = \sum_m \frac{\|f_m\|^2_{\mathcal{H}_m}}{w_m}$$

**Interprétation :** Le classifieur MKL apprend simultanément :
- **Quelles structures** sont importantes (via les poids $w_m$)
- **Comment classer** dans chaque espace (via les $f_m$)

### Contraintes sur les poids

| Contrainte | Formulation | Interprétation |
|------------|-------------|----------------|
| $\sum_m w_m = 1, w_m \geq 0$ | Simplexe | Interpolation convexe des kernels |
| $\|w\|_2 \leq 1, w_m \geq 0$ | Boule L2 | Régularisation plus douce |
| $w_m \in \{0,1\}$ | Binaire | Sélection de sous-ensemble (QUBO !) |

Notre projet QMKL utilise les deux : **Centered Alignment** (simplexe) et **QUBO** (binaire).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.datasets import make_moons
from sklearn.preprocessing import MinMaxScaler, StandardScaler

np.random.seed(42)

def K_Z(X, alpha=1.0):
    n, d = X.shape
    K = np.ones((n,n))
    for k in range(d):
        diff = X[:,k:k+1] - X[:,k].reshape(1,-1)
        K *= np.cos(alpha*diff)**2
    return K

def K_ZZ(X, alpha=1.0):
    K = K_Z(X, alpha)
    n, d = X.shape
    for k in range(d):
        for l in range(k+1, d):
            c = X[:,k]*X[:,l]
            diff = c.reshape(-1,1) - c.reshape(1,-1)
            K *= np.cos(alpha*diff)**2
    return K

def K_rbf(X, gamma=1.0):
    sq = np.sum(X**2, axis=1, keepdims=True)
    return np.exp(-gamma*(sq + sq.T - 2*(X@X.T)))

# ── Montrer que K_mkl est PSD pour tout w ≥ 0 ────────────────────────────
n = 20
X_test = np.random.uniform(0, 2, (n, 4))

K1 = K_Z(X_test, alpha=1.0)
K2 = K_ZZ(X_test, alpha=2.0)
K3 = K_rbf(X_test, gamma=1.0)

print("Vérification PSD des kernels individuels:")
for K, name in [(K1,'K_Z'),(K2,'K_ZZ'),(K3,'K_RBF')]:
    eigs = np.linalg.eigvalsh(K)
    print(f"  {name}: min eigenvalue = {eigs.min():.6f} → PSD: {eigs.min() > -1e-9}")

print("
Vérification PSD des combinaisons pondérées:")
for w in [(1/3, 1/3, 1/3), (0.7, 0.2, 0.1), (0.0, 0.5, 0.5), (1.0, 0.0, 0.0)]:
    K_mkl = w[0]*K1 + w[1]*K2 + w[2]*K3
    eigs = np.linalg.eigvalsh(K_mkl)
    print(f"  w={w}: min eigenvalue = {eigs.min():.6f} → PSD: {eigs.min() > -1e-9}")

# ── Effet des poids sur l'AUC ─────────────────────────────────────────────
X_moons, y_moons = make_moons(n_samples=100, noise=0.15, random_state=42)
X_moons = MinMaxScaler(feature_range=(0,2)).fit_transform(
              StandardScaler().fit_transform(X_moons))
X_moons = np.hstack([X_moons, np.zeros((len(X_moons), 2))])  # compléter à 4 dims
y_pm = 2*y_moons - 1

K_a = K_Z(X_moons, alpha=1.0)
K_b = K_ZZ(X_moons, alpha=2.0)
K_c = K_rbf(X_moons[:,:2], gamma=1.0)

# Grille de poids w1, w2 (w3 = 1 - w1 - w2)
n_pts = 20
w1_vals = np.linspace(0, 1, n_pts)
w2_vals = np.linspace(0, 1, n_pts)
AUC_grid = np.full((n_pts, n_pts), np.nan)

for i, w1 in enumerate(w1_vals):
    for j, w2 in enumerate(w2_vals):
        w3 = 1 - w1 - w2
        if w3 < 0: continue
        K_mkl = w1*K_a + w2*K_b + w3*K_c
        eigs = np.linalg.eigvalsh(K_mkl)
        if eigs.min() < -1e-8:
            K_mkl += (-eigs.min()+1e-8)*np.eye(len(K_mkl))
        try:
            auc = cross_val_score(SVC(kernel='precomputed',C=1.0), K_mkl, y_pm,
                                  cv=4, scoring='roc_auc').mean()
            AUC_grid[j, i] = auc
        except: pass

# AUC kernels individuels
aucs_indiv = {}
for K, name in [(K_a,'Z α=1'),(K_b,'ZZ α=2'),(K_c,'RBF γ=1')]:
    eigs = np.linalg.eigvalsh(K)
    if eigs.min()<0: K += (-eigs.min()+1e-8)*np.eye(len(K))
    aucs_indiv[name] = cross_val_score(SVC(kernel='precomputed',C=1.0), K, y_pm,
                                       cv=4, scoring='roc_auc').mean()

fig, ax = plt.subplots(figsize=(8, 6))
cf = ax.contourf(w1_vals, w2_vals, AUC_grid, levels=15, cmap='RdYlGn',
                 vmin=0.5, vmax=1.0)
plt.colorbar(cf, ax=ax, label='AUC (4-fold)')
ax.contour(w1_vals, w2_vals, AUC_grid, levels=[0.8, 0.85, 0.90], colors='black',
           linewidths=1, alpha=0.7)
# Marqueurs kernels individuels
ax.scatter(1.0, 0.0, s=150, color='#3498db', zorder=5,
           label=f'K_Z seul AUC={aucs_indiv["Z α=1"]:.3f}')
ax.scatter(0.0, 1.0, s=150, color='#e74c3c', marker='s', zorder=5,
           label=f'K_ZZ seul AUC={aucs_indiv["ZZ α=2"]:.3f}')
ax.scatter(0.0, 0.0, s=150, color='#2ecc71', marker='^', zorder=5,
           label=f'K_RBF seul AUC={aucs_indiv["RBF γ=1"]:.3f}')
# Optimum MKL
best_i, best_j = np.unravel_index(np.nanargmax(AUC_grid), AUC_grid.shape)
ax.scatter(w1_vals[best_i], w2_vals[best_j], s=200, color='gold', marker='*',
           zorder=6, label=f'MKL optimal AUC={np.nanmax(AUC_grid):.3f}')
ax.set_xlabel('w₁ (poids K_Z)', fontsize=11)
ax.set_ylabel('w₂ (poids K_ZZ)', fontsize=11)
ax.set_title('Paysage AUC(w₁, w₂) sur dataset Lunes
'
             'w₃ = 1 - w₁ - w₂ (poids K_RBF)', fontweight='bold')
ax.legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.show()

print(f"
Meilleur MKL: w1={w1_vals[best_i]:.2f}, w2={w2_vals[best_j]:.2f}, "
      f"w3={1-w1_vals[best_i]-w2_vals[best_j]:.2f}")
print(f"Gain MKL vs meilleur individuel: "
      f"+{np.nanmax(AUC_grid) - max(aucs_indiv.values()):.3f} AUC")


## Section 3 — Average : la baseline naïve

### Formule

$$K_{\text{avg}} = \frac{1}{M} \sum_{m=1}^M K_m$$

Poids uniformes $w_m = 1/M$. Aucun paramètre à ajuster.

### Quand Average fonctionne

Average est efficace quand :
1. **Tous les kernels ont une AUC similaire** — aucun kernel n'est clairement meilleur
2. **Les kernels sont diversifiés** — en moyennant, on réduit la variance des prédictions
3. **$N$ est grand** — la loi des grands nombres aide la combinaison à converger

**Analogue en finance :** La diversification de portefeuille. Combiner des actifs non-corrélés réduit la variance sans réduire l'espérance de rendement.

### Quand Average échoue

Average est mauvais quand :
1. **Certains kernels sont mauvais (AUC < 0.5)** — ils dégradent la combinaison
2. **$N$ est très petit** — le bruit statistique est amplifié par la moyenne
3. **Les kernels sont tous très similaires (redondants)** — moyenne ≈ un seul kernel, pas d'amélioration

### Résultat empirique

Dans notre projet : Average = $0.635$ AUC sur German Credit ($N=40$), vs Centered Alignment = $0.743$.
**Écart de $-10$ pts** ! La dilution par les kernels peu informatifs (AUC < 0.65) est très coûteuse quand $N$ est petit.


## Section 4 — Centered Alignment : dérivation complète

### Objectif

On veut trouver les poids $\mathbf{w}$ qui maximisent la corrélation entre $K_{\text{mkl}}$ et la matrice cible $\mathbf{Y} = \mathbf{y}\mathbf{y}^\top$ :
$$\max_{\mathbf{w}} \hat{A}\left(\sum_m w_m K_m, \mathbf{Y}\right)$$

avec $w_m \geq 0$ (pas de contrainte $\sum = 1$ dans la formulation original, mais on normalise ensuite).

### Étape 1 : Linéariser le numérateur

$$\left\langle \sum_m w_m \tilde{K}_m, \tilde{\mathbf{Y}} \right\rangle_F = \sum_m w_m \langle \tilde{K}_m, \tilde{\mathbf{Y}} \rangle_F = \sum_m w_m q_m$$

où $q_m = \langle \tilde{K}_m, \tilde{\mathbf{Y}} \rangle_F$ est l'alignement individuel du kernel $m$ avec la cible. (Le tilde indique le centrage.)

**Vecteur d'alignements individuels** : $\mathbf{q} = (q_1, ..., q_M)^\top \in \mathbb{R}^M$.

### Étape 2 : Quadratiser le dénominateur

$$\left\|\sum_m w_m \tilde{K}_m\right\|_F^2 = \sum_{m,m'} w_m w_{m'} \langle \tilde{K}_m, \tilde{K}_{m'} \rangle_F = \mathbf{w}^\top \mathbf{S} \mathbf{w}$$

où $S_{mm'} = \langle \tilde{K}_m, \tilde{K}_{m'} \rangle_F$ est la **matrice d'alignement mutuel** entre kernels.

### Étape 3 : Problème d'optimisation

$$\max_{\mathbf{w} \geq 0} \frac{\mathbf{w}^\top \mathbf{q}}{\sqrt{\mathbf{w}^\top \mathbf{S} \mathbf{w}} \cdot \|\tilde{\mathbf{Y}}\|_F}$$

C'est équivalent à (en ignorant $\|\tilde{\mathbf{Y}}\|_F$ constant) :
$$\max_{\mathbf{w} \geq 0, \mathbf{w}^\top \mathbf{S} \mathbf{w} \leq 1} \mathbf{w}^\top \mathbf{q}$$

### Étape 4 : Solution

En utilisant les conditions KKT, la solution optimale est :
$$\mathbf{w}^* = \frac{\mathbf{S}^{-1} \mathbf{q}}{\|\mathbf{S}^{-1} \mathbf{q}\|} \quad (\text{normalisée})$$

puis on projette sur le simplexe en mettant à zéro les composantes négatives et en renormalisant.

**Complexité :** $O(M^3)$ pour l'inversion de $\mathbf{S}$, mais $M \leq 20$ en pratique → très rapide.

**Clé :** Pas de processus itératif, pas d'hyperparamètre. Solution **analytique fermée**.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.datasets import make_moons, load_breast_cancer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA

np.random.seed(42)

# ── Fonctions kernel et centrage ─────────────────────────────────────────
def K_Z(X, alpha=1.0):
    n, d = X.shape
    K = np.ones((n,n))
    for k in range(d):
        diff = X[:,k:k+1] - X[:,k].reshape(1,-1)
        K *= np.cos(alpha*diff)**2
    return K

def K_ZZ(X, alpha=1.0):
    K = K_Z(X, alpha)
    n, d = X.shape
    for k in range(d):
        for l in range(k+1, d):
            c = X[:,k]*X[:,l]
            diff = c.reshape(-1,1) - c.reshape(1,-1)
            K *= np.cos(alpha*diff)**2
    return K

def center_kernel(K):
    n = len(K)
    H = np.eye(n) - np.ones((n,n))/n
    return H @ K @ H

def frobenius_ip(A, B):
    return np.sum(A * B)

def centered_alignment_mkl(Kmats, y):
    '''
    Centered Alignment MKL.
    Retourne les poids optimaux w*.

    Étapes:
    1. Centrer tous les kernels
    2. Calculer q_m = <K_m_c, Y_c>_F
    3. Calculer S_mm' = <K_m_c, K_m'_c>_F
    4. Résoudre: w* = S^{-1} q, puis projeter sur w >= 0 et normaliser
    '''
    n = len(Kmats[0])
    M = len(Kmats)
    Y = np.outer(y, y)
    Yc = center_kernel(Y)

    # Étape 1 : Centrer tous les kernels
    Kmats_c = [center_kernel(K) for K in Kmats]

    # Étape 2 : Calculer le vecteur q (alignements individuels)
    q = np.array([frobenius_ip(Kc, Yc) for Kc in Kmats_c])

    # Étape 3 : Calculer la matrice S (alignements mutuels)
    S = np.array([[frobenius_ip(Kmats_c[m], Kmats_c[mp])
                   for mp in range(M)] for m in range(M)])

    print(f"  q (alignements individuels): {q.round(1)}")
    print(f"  S diagonale (norme²): {np.diag(S).round(1)}")
    print(f"  Conditionnement de S: {np.linalg.cond(S):.1f}")

    # Étape 4 : Résoudre w* = S^{-1} q
    try:
        # Régulariser légèrement pour stabilité numérique
        S_reg = S + 1e-8 * np.eye(M)
        w = np.linalg.solve(S_reg, q)
    except np.linalg.LinAlgError:
        w = np.linalg.lstsq(S, q, rcond=None)[0]

    # Projeter sur w >= 0 et normaliser sur le simplexe
    w = np.maximum(w, 0)
    if w.sum() > 1e-10:
        w = w / w.sum()
    else:
        w = np.ones(M) / M  # fallback: average

    return w, q, S

# ── Expérience sur Breast Cancer ────────────────────────────────────────
data = load_breast_cancer()
X_raw, y_raw = data.data, data.target
y_pm = 2*y_raw - 1

X_prep = MinMaxScaler(feature_range=(0,2)).fit_transform(
          PCA(n_components=4).fit_transform(
          StandardScaler().fit_transform(X_raw)))

# Sous-ensemble de 150 points
idx = np.random.choice(len(X_prep), 150, replace=False)
X, y = X_prep[idx], y_pm[idx]

# Définir M=6 kernels
kernel_defs = [
    ('Z α=1.0',  K_Z,  1.0),
    ('Z α=3.0',  K_Z,  3.0),
    ('ZZ α=1.0', K_ZZ, 1.0),
    ('ZZ α=2.5', K_ZZ, 2.5),
    ('ZZ α=4.0', K_ZZ, 4.0),
    ('Z α=0.5',  K_Z,  0.5),
]

Kmats = []
for name, fn, alpha in kernel_defs:
    K = fn(X, alpha)
    np.fill_diagonal(K, 1.0)
    Kmats.append(K)

print("=== Centered Alignment MKL ===")
w_opt, q_vals, S_mat = centered_alignment_mkl(Kmats, y)

print(f"
Poids optimaux w*:")
for (name, _, _), w in zip(kernel_defs, w_opt):
    print(f"  {name}: w = {w:.4f}")

# ── Comparer les méthodes ────────────────────────────────────────────────
def eval_kernel(K, y, cv=5):
    eigs = np.linalg.eigvalsh(K)
    if eigs.min() < 0:
        K = K + (-eigs.min()+1e-8)*np.eye(len(K))
    return cross_val_score(SVC(kernel='precomputed', C=1.0), K, y,
                           cv=cv, scoring='roc_auc').mean()

# AUC individuels
print("
=== AUC par kernel ===")
aucs_indiv = []
for name, fn, alpha in kernel_defs:
    K = fn(X, alpha)
    np.fill_diagonal(K, 1.0)
    auc = eval_kernel(K, y)
    aucs_indiv.append(auc)
    print(f"  {name}: AUC = {auc:.4f}")

# Average
K_avg = np.mean(Kmats, axis=0)
auc_avg = eval_kernel(K_avg, y)
print(f"
  Average (1/M): AUC = {auc_avg:.4f}")

# Centered Alignment
K_ca = sum(w * K for w, K in zip(w_opt, Kmats))
auc_ca = eval_kernel(K_ca, y)
print(f"  Centered Align.: AUC = {auc_ca:.4f}")

# ── Visualisation ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
names = [n for n,_,_ in kernel_defs]
colors_bar = ['#3498db','#3498db','#e74c3c','#e74c3c','#e74c3c','#3498db']
bars = ax.barh(names, aucs_indiv, color=colors_bar, alpha=0.8, height=0.6)
for bar, auc in zip(bars, aucs_indiv):
    ax.text(auc+0.002, bar.get_y()+bar.get_height()/2,
            f'{auc:.3f}', va='center', fontsize=9)
ax.axvline(auc_avg, color='#f39c12', lw=2.5, ls='--', label=f'Average={auc_avg:.3f}')
ax.axvline(auc_ca,  color='#2ecc71', lw=2.5, label=f'Centered Align.={auc_ca:.3f}')
ax.set_xlabel('AUC (5-fold cross-val)')
ax.set_title('AUC par kernel vs méthodes MKL
(Breast Cancer, N=150, Q=4)', fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim(0.85, 1.01)

ax2 = axes[1]
# Heatmap de S (alignements mutuels)
im = ax2.imshow(S_mat / np.max(S_mat), cmap='Reds', vmin=0, vmax=1)
plt.colorbar(im, ax=ax2, fraction=0.046)
ax2.set_xticks(range(len(names))), ax2.set_yticks(range(len(names)))
ax2.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
ax2.set_yticklabels(names, fontsize=8)
ax2.set_title('Matrice S (alignements mutuels normalisés)
'
              'Rouge = kernels très similaires (redondants)', fontweight='bold')
for i in range(len(names)):
    for j in range(len(names)):
        ax2.text(j, i, f'{S_mat[i,j]/np.max(S_mat):.2f}',
                 ha='center', va='center', fontsize=8)

plt.tight_layout()
plt.show()

print(f"
Gain Centered Alignment vs Average: +{auc_ca - auc_avg:.4f}")
print(f"Gain Centered Alignment vs meilleur individuel: "
      f"{auc_ca - max(aucs_indiv):+.4f}")


## Section 5 — Quand MKL bat-il un kernel unique ?

### La condition de complémentarité

**Théorème (empirique) :** MKL apporte une amélioration sur le meilleur kernel individuel si et seulement si :
1. **Les kernels sont complémentaires** : $A(K_m, K_{m'}) < A(K_m, K_m)$ (faible alignement mutuel)
2. **Chaque kernel capture une information différente** de $\mathbf{y}\mathbf{y}^\top$

**Mesure de diversité :** $\text{Div}(\mathcal{K}) = 1 - \frac{1}{M(M-1)} \sum_{m \neq m'} \hat{A}(K_m, K_{m'})$

Une diversité proche de 1 = kernels très différents = fort potentiel MKL.

### Scénarios

| Scénario | Diversité | Gain MKL attendu |
|----------|-----------|-----------------|
| Tous identiques ($K_m = K$) | $0$ | $0$ (MKL = kernel seul) |
| Variantes du même type (RBF, différents $\gamma$) | $0.1-0.3$ | Faible ($+1-2$ pts) |
| Familles différentes (Z, ZZ, RBF) | $0.4-0.7$ | Modéré ($+3-5$ pts) |
| Structures très différentes (local, global, périodique) | $0.7-1.0$ | Fort ($+5-10$ pts) |

### Ce qu'on observe dans notre projet

Sur les données financières :
- Les 12 kernels quantiques de même famille sont **redondants** (diversité faible)
- Kernels de familles différentes (Z vs YZX) ont une **diversité modérée**
- **QUBO** (Notebook D) résout le problème en sélectionnant les 2 kernels les plus complémentaires


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.datasets import make_moons
from sklearn.preprocessing import MinMaxScaler, StandardScaler

np.random.seed(42)

def K_Z(X, alpha=1.0):
    n, d = X.shape
    K = np.ones((n,n))
    for k in range(d):
        diff = X[:,k:k+1] - X[:,k].reshape(1,-1)
        K *= np.cos(alpha*diff)**2
    return K

def K_ZZ(X, alpha=1.0):
    K = K_Z(X, alpha)
    n, d = X.shape
    for k in range(d):
        for l in range(k+1, d):
            c = X[:,k]*X[:,l]
            diff = c.reshape(-1,1) - c.reshape(1,-1)
            K *= np.cos(alpha*diff)**2
    return K

def K_rbf(X, gamma=1.0):
    sq = np.sum(X**2, axis=1, keepdims=True)
    return np.exp(-gamma*(sq+sq.T-2*(X@X.T)))

def center_kernel(K):
    n = len(K)
    H = np.eye(n) - np.ones((n,n))/n
    return H @ K @ H

def frobenius_ip(A, B): return np.sum(A*B)

def centered_alignment(K, y):
    Kc = center_kernel(K)
    Yc = center_kernel(np.outer(y,y))
    num = frobenius_ip(Kc, Yc)
    den = np.sqrt(frobenius_ip(Kc,Kc) * frobenius_ip(Yc,Yc))
    return num/den if den>0 else 0.

def mutual_alignment(K1, K2):
    K1c, K2c = center_kernel(K1), center_kernel(K2)
    num = frobenius_ip(K1c, K2c)
    den = np.sqrt(frobenius_ip(K1c,K1c) * frobenius_ip(K2c,K2c))
    return num/den if den>0 else 0.

def average_mkl(Kmats): return np.mean(Kmats, axis=0)

def centered_align_mkl(Kmats, y):
    M = len(Kmats)
    Y = np.outer(y,y)
    Yc = center_kernel(Y)
    Kmats_c = [center_kernel(K) for K in Kmats]
    q = np.array([frobenius_ip(Kc, Yc) for Kc in Kmats_c])
    S = np.array([[frobenius_ip(Kmats_c[m], Kmats_c[mp]) for mp in range(M)] for m in range(M)])
    S_reg = S + 1e-8*np.eye(M)
    try:
        w = np.linalg.solve(S_reg, q)
    except:
        w = np.ones(M)/M
    w = np.maximum(w, 0)
    return w/w.sum() if w.sum()>1e-10 else np.ones(M)/M

def eval_K(K, y, cv=5):
    eigs = np.linalg.eigvalsh(K)
    if eigs.min() < 0: K = K + (-eigs.min()+1e-8)*np.eye(len(K))
    return cross_val_score(SVC(kernel='precomputed',C=1.), K, y,
                           cv=cv, scoring='roc_auc').mean()

# ── Données ───────────────────────────────────────────────────────────────
X_raw, y_raw = make_moons(n_samples=120, noise=0.15, random_state=42)
X = MinMaxScaler(feature_range=(0,2)).fit_transform(StandardScaler().fit_transform(X_raw))
X = np.hstack([X, np.zeros((len(X), 2))])  # pad à 4 dims
y = 2*y_raw - 1

# ── 4 scénarios de diversité ─────────────────────────────────────────────
scenarios = {
    'Kernels identiques': [K_Z(X, 1.0)] * 4,
    'Même famille (Z)': [K_Z(X, a) for a in [0.5, 1.0, 2.0, 3.0]],
    'Familles différentes': [K_Z(X,1.0), K_ZZ(X,1.0), K_rbf(X[:,:2],1.0), K_ZZ(X,2.5)],
    'Max diversité': [K_Z(X,0.5), K_ZZ(X,3.0), K_rbf(X[:,:2],0.1), K_rbf(X[:,:2],5.0)],
}

results = {}
for sc_name, Kmats in scenarios.items():
    # AUC individuels
    aucs_ind = [eval_K(K, y) for K in Kmats]
    best_ind = max(aucs_ind)

    # Average
    K_avg = average_mkl(Kmats)
    auc_avg = eval_K(K_avg, y)

    # Centered Alignment
    w_ca = centered_align_mkl(Kmats, y)
    K_ca = sum(w*K for w,K in zip(w_ca, Kmats))
    auc_ca = eval_K(K_ca, y)

    # Diversité (1 - moyenne alignements mutuels)
    M = len(Kmats)
    aligns = [mutual_alignment(Kmats[m], Kmats[mp])
              for m in range(M) for mp in range(M) if m != mp]
    diversity = 1 - np.mean(aligns)

    results[sc_name] = {
        'best_ind': best_ind, 'auc_avg': auc_avg, 'auc_ca': auc_ca,
        'diversity': diversity,
        'gain_ca_vs_ind': auc_ca - best_ind,
    }
    print(f"
{sc_name}:")
    print(f"  Diversité = {diversity:.3f}")
    print(f"  Meilleur individuel: {best_ind:.4f}")
    print(f"  Average: {auc_avg:.4f} ({auc_avg-best_ind:+.4f})")
    print(f"  Centered Align.: {auc_ca:.4f} ({auc_ca-best_ind:+.4f})")

# ── Graphique résumé ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sc_names = list(results.keys())
x_pos = np.arange(len(sc_names))

ax = axes[0]
ax.bar(x_pos - 0.25, [results[s]['best_ind'] for s in sc_names],
       0.25, label='Meilleur individuel', color='#3498db', alpha=0.8)
ax.bar(x_pos, [results[s]['auc_avg'] for s in sc_names],
       0.25, label='Average MKL', color='#f39c12', alpha=0.8)
ax.bar(x_pos + 0.25, [results[s]['auc_ca'] for s in sc_names],
       0.25, label='Centered Align.', color='#2ecc71', alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(sc_names, fontsize=8, rotation=10)
ax.set_ylabel('AUC')
ax.set_title('AUC selon le scénario de diversité
(Dataset Lunes, N=120)', fontweight='bold')
ax.legend(fontsize=8)
ax.set_ylim(0.7, 1.05)
ax.grid(axis='y', alpha=0.3)

ax2 = axes[1]
diversities = [results[s]['diversity'] for s in sc_names]
gains = [results[s]['gain_ca_vs_ind'] for s in sc_names]
ax2.bar(sc_names, gains, color=['#27ae60' if g>0 else '#c0392b' for g in gains], alpha=0.85)
ax2.axhline(0, color='black', lw=1.5)
for i, (d, g) in enumerate(zip(diversities, gains)):
    ax2.text(i, g + 0.003 * (1 if g >= 0 else -1),
             f'Div={d:.2f}', ha='center', fontsize=9, fontweight='bold')
ax2.set_ylabel('Gain AUC (Centered Align. - meilleur individuel)')
ax2.set_title('Gain MKL vs diversité des kernels
'
              'Plus les kernels sont diversifiés, plus le gain est grand', fontweight='bold')
ax2.tick_params(axis='x', labelsize=8, rotation=10)

plt.tight_layout()
plt.show()

print("
→ Conclusion: MKL n'aide QUE si les kernels sont diversifiés.")
print("→ Combiner des copies du même kernel = 0 gain.")
